In [1]:
import pandas as pd
import os

In [2]:
path_pickle = r'/Users/mariatirado/29-10-2025 Instacart Basket Analysis/02 Data/Prepared Data'

In [3]:
# Importing main dataframe
ords_prods_merge = pd.read_pickle(os.path.join(path_pickle, 'ords_prods_merge_updated.pkl'))

# Quick check to confirm it's loaded
ords_prods_merge.shape

(32432460, 18)

In [4]:
# Create subset
df = ords_prods_merge[:1000000]

# Confirm size
print('Subset shape:', df.shape)

Subset shape: (1000000, 18)


In [5]:
df.head(10)

,order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,add_to_cart_order,reordered,product_name,aisle_id,department_id,prices,price_range_loc,day_type,busiest_days_new,busiest_period_of_day
0,2539329,1,prior,1,2,8,NaN,196,1,0,Soda,77,7,9.0,Mid-range product,Regularly busy,Regularly busy,Average orders
1,2539329,1,prior,1,2,8,NaN,14084,2,0,Organic Unsweetened Vanilla Almond Milk,91,16,12.5,Mid-range product,Regularly busy,Regularly busy,Average orders
2,2539329,1,prior,1,2,8,NaN,12427,3,0,Original Beef Jerky,23,19,4.4,Low-range product,Regularly busy,Regularly busy,Average orders
3,2539329,1,prior,1,2,8,NaN,26088,4,0,Aged White Cheddar Popcorn,23,19,4.7,Low-range product,Regularly busy,Regularly busy,Average orders
4,2539329,1,prior,1,2,8,NaN,26405,5,0,XL Pick-A-Size Paper Towel Rolls,54,17,1.0,Low-range product,Regularly busy,Regularly busy,Average orders
5,2398795,1,prior,2,3,7,15.0,196,1,1,Soda,77,7,9.0,Mid-range product,Regularly busy,Slowest days,Average orders
6,2398795,1,prior,2,3,7,15.0,10258,2,0,Pistachios,117,19,3.0,Low-range product,Regularly busy,Slowest days,Average orders
7,2398795,1,prior,2,3,7,15.0,12427,3,1,Original Beef Jerky,23,19,4.4,Low-range product,Regularly busy,Slowest days,Average orders
8,2398795,1,prior,2,3,7,15.0,13176,4,0,Bag of Organic Bananas,24,4,10.3,Mid-range product,Regularly busy,Slowest days,Average orders
9,2398795,1,prior,2,3,7,15.0,26088,5,1,Aged White Cheddar Popcorn,23,19,4.7,Low-range product,Regularly busy,Slowest days,Average orders


In [6]:
df.groupby('product_name')

In [7]:
df.groupby('department_id').agg({'order_number': ['mean']})

,order_number
,mean
department_id,
1,14.803399
2,17.091743
3,17.930716
4,17.897312
5,15.214270
6,15.382228
7,17.700476
8,16.485269


In [8]:
df.groupby('department_id').order_number.mean()

department_id
1     14.803399
2     17.091743
3     17.930716
4     17.897312
5     15.214270
6     15.382228
7     17.700476
8     16.485269
9     15.965921
10    20.091818
11    16.484474
12    15.619765
13    16.486000
14    17.508672
15    15.695987
16    18.009055
17    16.155822
18    19.606536
19    17.631556
20    17.140859
21    22.014421
Name: order_number, dtype: float64

# Performing Multiple Aggregations

In [9]:
df.groupby('department_id').agg({'order_number': ['mean', 'min', 'max']})

order_number        
                      mean min max
department_id                     
1                14.803399   1  99
2                17.091743   1  98
3                17.930716   1  99
4                17.897312   1  99
5                15.214270   1  99
6                15.382228   1  99
7                17.700476   1  99
8                16.485269   1  91
9                15.965921   1  99
10               20.091818   1  99
11               16.484474   1  99
12               15.619765   1  99
13               16.486000   1  99
14               17.508672   1  99
15               15.695987   1  99
16               18.009055   1  99
17               16.155822   1  99
18               19.606536   1  99
19               17.631556   1  99
20               17.140859   1  99
21               22.014421   1  97

# Aggregating Data with transform()

In [10]:
import numpy as np
import pandas as pd

In [12]:
ords_prods_merge['max_order'] = (
    ords_prods_merge
      .groupby('user_id')['order_number']
      .transform('max'))

In [13]:
ords_prods_merge[['user_id','order_number','max_order']].head()

,user_id,order_number,max_order
0,1,1,10
1,1,1,10
2,1,1,10
3,1,1,10
4,1,1,10


# Loyalty_flag

In [15]:
conds = [
    ords_prods_merge['max_order'] > 40,
    (ords_prods_merge['max_order'] > 10) & (ords_prods_merge['max_order'] <= 40),
    ords_prods_merge['max_order'] <= 10
]
labels = ['Loyal customer', 'Regular customer', 'New customer']

ords_prods_merge['loyalty_flag'] = np.select(
    conds,
    labels,
    default='Unknown')

ords_prods_merge['loyalty_flag'].value_counts(dropna=False)

loyalty_flag
Regular customer    15890123
Loyal customer      10293366
New customer         6248971
Name: count, dtype: int64